In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import joblib

In [2]:
DATA_DIR = Path("../PS3/02_Datasets/Door")

TEST_PATH = DATA_DIR / "Test.csv"
MODEL_PATH = Path("door_final_model.joblib")
FEATURE_PATH = Path("door_final_features.csv")

print("Test exists:", TEST_PATH.exists())
print("Model exists:", MODEL_PATH.exists())
print("Feature list exists:", FEATURE_PATH.exists())

Test exists: True
Model exists: True
Feature list exists: True


In [3]:
# Load the complete unlabelled Door test stream.

test = pd.read_csv(TEST_PATH)

print("Test shape:", test.shape)
display(test.head())

Test shape: (6253, 17)


,Datetime,Motor current(mA),Motor Voltage(10mV),Motor electrodynamic force,Door opening time(.1s),Door closing time(.1s),Close command,Open command,DCSR,DCSL,DLSR,DLSL,Door Opened,Door Locked,Door is opening,Door is closing,Door leaf position
0,2023-7-5-0-0-0-0,121,400,64,23,35,1,0,0,0,0,0,0,0,0,1,700
1,2023-7-5-0-0-0-20,150,600,112,23,35,1,0,0,0,0,0,0,0,0,1,700
2,2023-7-5-0-0-0-40,232,800,121,23,35,1,0,0,0,0,0,0,0,0,1,699
3,2023-7-5-0-0-0-60,409,1200,134,23,35,1,0,0,0,0,0,0,0,0,1,699
4,2023-7-5-0-0-0-80,588,1700,173,23,35,1,0,0,0,0,0,0,0,0,1,698


In [4]:
# Confirm that the test file uses the same raw columns expected by the training pipeline.

print("Test columns:")

for i, column in enumerate(test.columns):
    print(f"{i:2d}: {column}")

Test columns:
 0: Datetime
 1: Motor current(mA)
 2: Motor Voltage(10mV)
 3: Motor electrodynamic force
 4: Door opening time(.1s)
 5: Door closing time(.1s)
 6: Close command
 7: Open command
 8: DCSR
 9: DCSL
10: DLSR
11: DLSL
12: Door Opened
13: Door Locked
14: Door is opening
15: Door is closing
16: Door leaf position


In [5]:
# Check that all sensor columns required by the trained feature pipeline are available.

required_columns = ["Datetime", "Motor current(mA)", "Motor Voltage(10mV)", "Motor electrodynamic force", "Door leaf position"]

missing_columns = [column for column in required_columns if column not in test.columns]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

print("All required columns are present.")

All required columns are present.


In [6]:
# Convert the Door timestamp format into Pandas timestamps.


def parse_door_datetime(value):
    parts = str(value).split("-")

    if len(parts) != 7:
        return pd.NaT

    try:
        year, month, day, hour, minute, second, ms = map(int, parts)

        return pd.Timestamp(year=year, month=month, day=day, hour=hour, minute=minute, second=second, microsecond=ms * 1000)

    except (ValueError, TypeError):
        return pd.NaT

In [7]:
# Parse all Test.csv timestamps before performing temporal segmentation.

test["timestamp"] = test["Datetime"].apply(parse_door_datetime)

invalid_timestamps = test["timestamp"].isna().sum()

print("Invalid timestamps:", invalid_timestamps)

if invalid_timestamps > 0:
    raise ValueError("Test.csv contains invalid timestamps.")

Invalid timestamps: 0


In [8]:
# Calculate the time between consecutive Test.csv readings.

test["dt_seconds"] = test["timestamp"].diff().dt.total_seconds()

display(test["dt_seconds"].describe())

count    6252.000000
mean        0.228654
std         2.872710
min         0.020000
25%         0.020000
50%         0.020000
75%         0.020000
max        55.356000
Name: dt_seconds, dtype: float64

In [9]:
# Inspect the most common Test sampling intervals.

display(test["dt_seconds"].round(6).value_counts().head(20))

dt_seconds
0.020     6215
11.245       1
43.275       1
54.155       1
12.776       1
21.789       1
38.243       1
19.238       1
48.374       1
27.318       1
37.799       1
42.911       1
33.964       1
37.887       1
37.089       1
44.320       1
42.982       1
19.176       1
48.123       1
44.046       1
Name: count, dtype: int64

In [10]:
# Inspect the largest time gaps in the Test stream.

test_large_gaps = test[["Datetime", "timestamp", "dt_seconds"]].sort_values("dt_seconds", ascending=False).head(20)

display(test_large_gaps)

,Datetime,timestamp,dt_seconds
4400,2023-7-5-0-18-7-558,2023-07-05 00:18:07.558,55.356
519,2023-7-5-0-1-58-995,2023-07-05 00:01:58.995,54.155
3894,2023-7-5-0-15-27-662,2023-07-05 00:15:27.662,53.128
4067,2023-7-5-0-16-20-947,2023-07-05 00:16:20.947,49.845
6065,2023-7-5-0-23-45-804,2023-07-05 00:23:45.804,49.427
5613,2023-7-5-0-22-17-683,2023-07-05 00:22:17.683,48.399
1336,2023-7-5-0-4-35-655,2023-07-05 00:04:35.655,48.374
2868,2023-7-5-0-11-17-664,2023-07-05 00:11:17.664,48.123
4257,2023-7-5-0-17-9-362,2023-07-05 00:17:09.362,44.635
3517,2023-7-5-0-14-5-850,2023-07-05 00:14:05.850,44.619


In [11]:
# Count gaps above the segmentation threshold used during development.

GAP_THRESHOLD_SECONDS = 1.0

test_gap_indices = test.index[test["dt_seconds"] > GAP_THRESHOLD_SECONDS].tolist()

print("Large gaps detected:", len(test_gap_indices))
print("Predicted segments:", len(test_gap_indices) + 1)

Large gaps detected: 37
Predicted segments: 38


In [12]:
# Convert the detected timestamp gaps into individual Test segments.

test_starts = [0] + test_gap_indices

test_ends = [index - 1 for index in test_gap_indices] + [len(test) - 1]

test_segments = pd.DataFrame({"start_idx": test_starts, "end_idx": test_ends})

test_segments["n_rows"] = test_segments["end_idx"] - test_segments["start_idx"] + 1

test_segments["start_timestamp"] = test_segments["start_idx"].map(test["timestamp"])

test_segments["end_timestamp"] = test_segments["end_idx"].map(test["timestamp"])

test_segments["duration_seconds"] = (test_segments["end_timestamp"] - test_segments["start_timestamp"]).dt.total_seconds()

print("Test segments:", len(test_segments))

display(test_segments.head(10))

Test segments: 38


,start_idx,end_idx,n_rows,start_timestamp,end_timestamp,duration_seconds
0,0,188,189,2023-07-05 00:00:00.000,2023-07-05 00:00:03.760,3.76
1,189,377,189,2023-07-05 00:00:15.005,2023-07-05 00:00:18.765,3.76
2,378,518,141,2023-07-05 00:01:02.040,2023-07-05 00:01:04.840,2.80
3,519,658,140,2023-07-05 00:01:58.995,2023-07-05 00:02:01.775,2.78
4,659,834,176,2023-07-05 00:02:14.551,2023-07-05 00:02:18.051,3.50
5,835,1020,186,2023-07-05 00:02:39.840,2023-07-05 00:02:43.540,3.70
6,1021,1163,143,2023-07-05 00:03:21.783,2023-07-05 00:03:24.623,2.84
7,1164,1335,172,2023-07-05 00:03:43.861,2023-07-05 00:03:47.281,3.42
8,1336,1474,139,2023-07-05 00:04:35.655,2023-07-05 00:04:38.415,2.76
9,1475,1611,137,2023-07-05 00:05:05.733,2023-07-05 00:05:08.453,2.72


In [13]:
# Inspect the distribution of detected Test cycle lengths.

display(test_segments[["n_rows", "duration_seconds"]].describe())

,n_rows,duration_seconds
count,38.000000,38.000000
mean,164.552632,3.271053
std,22.521698,0.450434
min,135.000000,2.680000
25%,142.000000,2.820000
50%,174.000000,3.460000
75%,186.750000,3.715000
max,190.000000,3.780000


In [14]:
# Check that every detected segment has a valid temporal interval.

invalid_segments = test_segments[(test_segments["end_timestamp"] < test_segments["start_timestamp"]) | (test_segments["n_rows"] <= 0)]

print("Invalid segments:", len(invalid_segments))

if len(invalid_segments) > 0:
    display(invalid_segments)
    raise ValueError("Invalid Test segments detected.")

Invalid segments: 0


In [15]:
# Load the trained preprocessing + Logistic Regression pipeline.

final_model = joblib.load(MODEL_PATH)

print("Final model loaded.")
print(final_model)

Final model loaded.
Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))])


In [16]:
# Load the exact feature list used when training the final model.

final_features = pd.read_csv(FEATURE_PATH)["feature"].tolist()

print("Number of final features:", len(final_features))

for feature in final_features:
    print(feature)

Number of final features: 19
Motor current(mA)_mean
Motor current(mA)_std
Motor current(mA)_min
Motor current(mA)_max
Motor current(mA)_range
Motor Voltage(10mV)_mean
Motor Voltage(10mV)_std
Motor Voltage(10mV)_min
Motor Voltage(10mV)_max
Motor Voltage(10mV)_range
Motor electrodynamic force_mean
Motor electrodynamic force_std
Motor electrodynamic force_max
Motor electrodynamic force_range
Door leaf position_mean
Door leaf position_std
Door leaf position_min
Door leaf position_max
Door leaf position_range


In [17]:
# Calculate the same whole-cycle statistics used by the final classifier.


def basic_statistics(values, prefix):
    values = np.asarray(values, dtype=float)

    return {
        f"{prefix}_mean": np.mean(values),
        f"{prefix}_std": np.std(values),
        f"{prefix}_min": np.min(values),
        f"{prefix}_max": np.max(values),
        f"{prefix}_range": np.max(values) - np.min(values),
    }

In [18]:
# Extract one Test operation using its detected row boundaries.


def extract_cycle(data, segment):
    start_idx = int(segment["start_idx"])
    end_idx = int(segment["end_idx"])

    return data.loc[start_idx:end_idx].copy()

In [20]:
# Convert one segmented operation into the 19 features expected by the final model.


def calculate_final_features(cycle):
    features = {}

    for signal in ["Motor current(mA)", "Motor Voltage(10mV)", "Motor electrodynamic force", "Door leaf position"]:
        features.update(basic_statistics(cycle[signal], signal))

    return features

In [21]:
# Build the final feature matrix for every detected Test operation.

test_feature_rows = []

for _, segment in test_segments.iterrows():
    cycle = extract_cycle(test, segment)

    features = calculate_final_features(cycle)

    test_feature_rows.append(features)

test_features = pd.DataFrame(test_feature_rows)

print("Generated feature matrix:", test_features.shape)

display(test_features.head())

Generated feature matrix: (38, 20)


,Motor current(mA)_mean,Motor current(mA)_std,Motor current(mA)_min,Motor current(mA)_max,Motor current(mA)_range,Motor Voltage(10mV)_mean,Motor Voltage(10mV)_std,Motor Voltage(10mV)_min,Motor Voltage(10mV)_max,Motor Voltage(10mV)_range,Motor electrodynamic force_mean,Motor electrodynamic force_std,Motor electrodynamic force_min,Motor electrodynamic force_max,Motor electrodynamic force_range,Door leaf position_mean,Door leaf position_std,Door leaf position_min,Door leaf position_max,Door leaf position_range
0,463.978836,541.017960,0.0,2084.0,2084.0,4149.735450,1415.956847,400.0,6200.0,5800.0,878.243386,457.895027,0.0,1335.0,1335.0,286.433862,234.746091,1.0,700.0,699.0
1,463.116402,542.049313,0.0,2102.0,2102.0,4142.857143,1413.732456,400.0,6200.0,5800.0,877.624339,457.385082,0.0,1335.0,1335.0,286.523810,234.062750,2.0,700.0,698.0
2,660.248227,736.137593,6.0,2501.0,2495.0,5624.822695,3228.555131,300.0,9200.0,8900.0,1199.269504,918.889747,0.0,2287.0,2287.0,416.531915,262.087543,0.0,700.0,700.0
3,681.878571,758.146428,0.0,2501.0,2501.0,5716.428571,3336.285589,300.0,9800.0,9500.0,1212.000000,946.947004,0.0,2334.0,2334.0,417.771429,262.161900,0.0,699.0,699.0
4,462.250000,530.668302,0.0,2137.0,2137.0,4388.636364,1535.188160,200.0,6700.0,6500.0,948.738636,475.262200,0.0,1414.0,1414.0,294.164773,235.951413,0.0,700.0,700.0


In [22]:
# Ensure the generated Test features exactly match the features used during training.

generated_features = test_features.columns.tolist()

missing_features = [feature for feature in final_features if feature not in generated_features]

extra_features = [feature for feature in generated_features if feature not in final_features]

print("Missing final features:", missing_features)
print("Extra generated features:", extra_features)

if missing_features:
    raise ValueError("Test feature generation is missing required model features.")

Missing final features: []
Extra generated features: ['Motor electrodynamic force_min']


In [23]:
# Reorder the Test feature matrix to exactly match the training feature order.

test_features = test_features[final_features]

print("Final Test feature matrix:", test_features.shape)

Final Test feature matrix: (38, 19)


In [24]:
# Check for invalid feature values before inference.

print("NaN values:", test_features.isna().sum().sum())

print("Infinite values:", np.isinf(test_features.to_numpy()).sum())

if test_features.isna().sum().sum() > 0:
    raise ValueError("NaN values found in Test features.")

if np.isinf(test_features.to_numpy()).sum() > 0:
    raise ValueError("Infinite values found in Test features.")

NaN values: 0
Infinite values: 0


In [25]:
# Run every detected Test cycle through the saved final classifier.

predicted_labels = final_model.predict(test_features)

print("Predictions generated:", len(predicted_labels))

display(pd.Series(predicted_labels).value_counts())

Predictions generated: 38


0    29
1     9
Name: count, dtype: int64

In [26]:
# Generate model probabilities as a diagnostic output for our own analysis.

if hasattr(final_model, "predict_proba"):
    predicted_probabilities = final_model.predict_proba(test_features)

    class_names = final_model.classes_

    print("Model classes:", class_names)
    print("Probability matrix shape:", predicted_probabilities.shape)

Model classes: [0 1]
Probability matrix shape: (38, 2)


In [27]:
# Combine the detected boundaries and predicted labels into the required submission format.

submission = pd.DataFrame({"start_time": test_segments["start_timestamp"], "end_time": test_segments["end_timestamp"], "prediction": predicted_labels})

display(submission.head(10))

,start_time,end_time,prediction
0,2023-07-05 00:00:00.000,2023-07-05 00:00:03.760,0
1,2023-07-05 00:00:15.005,2023-07-05 00:00:18.765,0
2,2023-07-05 00:01:02.040,2023-07-05 00:01:04.840,0
3,2023-07-05 00:01:58.995,2023-07-05 00:02:01.775,0
4,2023-07-05 00:02:14.551,2023-07-05 00:02:18.051,0
5,2023-07-05 00:02:39.840,2023-07-05 00:02:43.540,0
6,2023-07-05 00:03:21.783,2023-07-05 00:03:24.623,0
7,2023-07-05 00:03:43.861,2023-07-05 00:03:47.281,0
8,2023-07-05 00:04:35.655,2023-07-05 00:04:38.415,0
9,2023-07-05 00:05:05.733,2023-07-05 00:05:08.453,0


In [28]:
# Convert timestamps into an ISO-compatible string representation for submission.

submission["start_time"] = submission["start_time"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f").str[:-3]

submission["end_time"] = submission["end_time"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f").str[:-3]

display(submission.head(10))

,start_time,end_time,prediction
0,2023-07-05T00:00:00.000,2023-07-05T00:00:03.760,0
1,2023-07-05T00:00:15.005,2023-07-05T00:00:18.765,0
2,2023-07-05T00:01:02.040,2023-07-05T00:01:04.840,0
3,2023-07-05T00:01:58.995,2023-07-05T00:02:01.775,0
4,2023-07-05T00:02:14.551,2023-07-05T00:02:18.051,0
5,2023-07-05T00:02:39.840,2023-07-05T00:02:43.540,0
6,2023-07-05T00:03:21.783,2023-07-05T00:03:24.623,0
7,2023-07-05T00:03:43.861,2023-07-05T00:03:47.281,0
8,2023-07-05T00:04:35.655,2023-07-05T00:04:38.415,0
9,2023-07-05T00:05:05.733,2023-07-05T00:05:08.453,0


In [29]:
# Validate the required Door submission columns.

expected_columns = ["start_time", "end_time", "prediction"]

print("Columns:", submission.columns.tolist())

if submission.columns.tolist() != expected_columns:
    raise ValueError(f"Unexpected submission columns: {submission.columns.tolist()}")

print("Column schema is valid.")

Columns: ['start_time', 'end_time', 'prediction']
Column schema is valid.


In [31]:
# Run the detected Test cycles through the final classifier and convert predictions back to submission labels.

predicted_binary = final_model.predict(test_features)

predicted_labels = np.where(predicted_binary == 1, "Abnormal resistance", "Normal")

print("Predictions generated:", len(predicted_labels))

display(pd.Series(predicted_labels).value_counts())

Predictions generated: 38


Normal                 29
Abnormal resistance     9
Name: count, dtype: int64

In [32]:
# Generate model probabilities as a diagnostic output for our own analysis.

if hasattr(final_model, "predict_proba"):
    predicted_probabilities = final_model.predict_proba(test_features)

    class_names = final_model.classes_

    print("Model classes:", class_names)
    print("Probability matrix shape:", predicted_probabilities.shape)

Model classes: [0 1]
Probability matrix shape: (38, 2)


In [33]:
# Combine the detected boundaries and predicted labels into the required submission format.

submission = pd.DataFrame({"start_time": test_segments["start_timestamp"], "end_time": test_segments["end_timestamp"], "prediction": predicted_labels})

display(submission.head(10))

,start_time,end_time,prediction
0,2023-07-05 00:00:00.000,2023-07-05 00:00:03.760,Normal
1,2023-07-05 00:00:15.005,2023-07-05 00:00:18.765,Normal
2,2023-07-05 00:01:02.040,2023-07-05 00:01:04.840,Normal
3,2023-07-05 00:01:58.995,2023-07-05 00:02:01.775,Normal
4,2023-07-05 00:02:14.551,2023-07-05 00:02:18.051,Normal
5,2023-07-05 00:02:39.840,2023-07-05 00:02:43.540,Normal
6,2023-07-05 00:03:21.783,2023-07-05 00:03:24.623,Normal
7,2023-07-05 00:03:43.861,2023-07-05 00:03:47.281,Normal
8,2023-07-05 00:04:35.655,2023-07-05 00:04:38.415,Normal
9,2023-07-05 00:05:05.733,2023-07-05 00:05:08.453,Normal


In [34]:
# Convert timestamps into an ISO-compatible string representation for submission.

submission["start_time"] = submission["start_time"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f").str[:-3]

submission["end_time"] = submission["end_time"].dt.strftime("%Y-%m-%dT%H:%M:%S.%f").str[:-3]

display(submission.head(10))

,start_time,end_time,prediction
0,2023-07-05T00:00:00.000,2023-07-05T00:00:03.760,Normal
1,2023-07-05T00:00:15.005,2023-07-05T00:00:18.765,Normal
2,2023-07-05T00:01:02.040,2023-07-05T00:01:04.840,Normal
3,2023-07-05T00:01:58.995,2023-07-05T00:02:01.775,Normal
4,2023-07-05T00:02:14.551,2023-07-05T00:02:18.051,Normal
5,2023-07-05T00:02:39.840,2023-07-05T00:02:43.540,Normal
6,2023-07-05T00:03:21.783,2023-07-05T00:03:24.623,Normal
7,2023-07-05T00:03:43.861,2023-07-05T00:03:47.281,Normal
8,2023-07-05T00:04:35.655,2023-07-05T00:04:38.415,Normal
9,2023-07-05T00:05:05.733,2023-07-05T00:05:08.453,Normal


In [35]:
# Validate the required Door submission columns.

expected_columns = ["start_time", "end_time", "prediction"]

print("Columns:", submission.columns.tolist())

if submission.columns.tolist() != expected_columns:
    raise ValueError(f"Unexpected submission columns: {submission.columns.tolist()}")

print("Column schema is valid.")

Columns: ['start_time', 'end_time', 'prediction']
Column schema is valid.


In [36]:
# Validate that every prediction uses an allowed Door status.

valid_predictions = {"Normal", "Abnormal resistance"}

invalid_predictions = set(submission["prediction"]) - valid_predictions

print("Invalid predictions:", invalid_predictions)

if invalid_predictions:
    raise ValueError("Invalid prediction label detected.")

print("Prediction labels are valid.")

Invalid predictions: set()
Prediction labels are valid.


In [37]:
# Check for missing values and duplicate predicted segments.

print("Missing values:", submission.isna().sum().sum())

duplicate_count = submission.duplicated().sum()

print("Duplicate rows:", duplicate_count)

if submission.isna().sum().sum() > 0:
    raise ValueError("Submission contains missing values.")

Missing values: 0
Duplicate rows: 0


In [38]:
# Confirm that predicted timestamps are chronologically valid.

start_times = pd.to_datetime(submission["start_time"])

end_times = pd.to_datetime(submission["end_time"])

invalid_intervals = (end_times < start_times).sum()

print("Invalid intervals:", invalid_intervals)

if invalid_intervals > 0:
    raise ValueError("Submission contains invalid time intervals.")

Invalid intervals: 0


In [39]:
# Summarise the number of Test operations assigned to each class.

prediction_counts = submission["prediction"].value_counts()

display(prediction_counts)

print("\nTotal predicted segments:", len(submission))

prediction
Normal                 29
Abnormal resistance     9
Name: count, dtype: int64


Total predicted segments: 38


In [41]:
OUTPUT_PATH = Path("door_predictions.csv")

submission.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH.resolve())

print("Rows:", len(submission))

Saved: /Users/yeo/Documents/Door/NOTEBOOKS/door_predictions.csv
Rows: 38


In [42]:
final_submission = pd.read_csv(OUTPUT_PATH)

print("Final submission shape:", final_submission.shape)

display(final_submission.head(20))

Final submission shape: (38, 3)


,start_time,end_time,prediction
0,2023-07-05T00:00:00.000,2023-07-05T00:00:03.760,Normal
1,2023-07-05T00:00:15.005,2023-07-05T00:00:18.765,Normal
2,2023-07-05T00:01:02.040,2023-07-05T00:01:04.840,Normal
3,2023-07-05T00:01:58.995,2023-07-05T00:02:01.775,Normal
4,2023-07-05T00:02:14.551,2023-07-05T00:02:18.051,Normal
5,2023-07-05T00:02:39.840,2023-07-05T00:02:43.540,Normal
6,2023-07-05T00:03:21.783,2023-07-05T00:03:24.623,Normal
7,2023-07-05T00:03:43.861,2023-07-05T00:03:47.281,Normal
8,2023-07-05T00:04:35.655,2023-07-05T00:04:38.415,Normal
9,2023-07-05T00:05:05.733,2023-07-05T00:05:08.453,Normal
